# LitNet — Ablation Study (Reviewed, Multi-Run Version)

**Purpose:** Systematically vary individual design choices of the LitUNet-DLU baseline and compare configurations using validation data only.

**Experimental protocol**
- Five independent training runs per configuration by default (`RUN_SEEDS = [42,  123,  2026,  3407,  5678]`).
- The same seed list is used for each configuration, enabling paired comparisons by seed.
- Results are stored as one row per configuration and seed. Checkpoints also include the seed in their filenames.
- Configurations are selected using validation results. The held-out test set is evaluated only after model selection.
- The notebook reports mean and standard deviation across runs and performs seed-matched paired tests with Holm correction for multiple comparisons.

**Important:** Existing first-run values were single-run exploratory results. They are not pooled with the new multi-run experiment because the experiment version and protocol differ. Existing CSVs are preserved as legacy files when incompatible with this experiment version.

**Ablation axes:** activation, loss, dropout, batch normalization, network width, DLU beta upper bound, optimizer (Adam/SGD/RMSprop), and Adam learning rate (1e-3/1e-4/1e-5).

In [1]:
# ============================================================
# 1. SETUP & IMPORTS (Colab)
# ============================================================
import os, gc, time, json, random, re
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from sklearn.metrics import precision_recall_fscore_support
from IPython.display import display, Javascript

# --- Colab keep-alive (auto-click reconnect) ---
display(Javascript('''
function ClickConnect(){
  console.log("keep-alive ping");
  document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(ClickConnect, 60000);
'''))

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

<IPython.core.display.Javascript object>

TF: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ============================================================
# MOUNT DRIVE + STAGE DATA LOCALLY
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ⚠️ EDIT THIS to match your actual Drive path
DRIVE_DATASET = "/content/drive/MyDrive/data_wound_seg"
# e.g. "/content/drive/MyDrive/Colab Notebooks/datasets/data_wound_seg"

LOCAL_DATASET = "/content/data_wound_seg"

# Copy once (fast because it's on Drive, only runs first time)
if not os.path.exists(LOCAL_DATASET):
    print("📥 Copying dataset from Drive to local disk...")
    !cp -r "{DRIVE_DATASET}" "{LOCAL_DATASET}"
    print("✅ Copied.")
else:
    print("✅ Local dataset already present (skipping copy).")

# Save ablation outputs back to Drive (so they survive Colab resets)
DRIVE_OUT = "/content/drive/MyDrive/litnet_ablation"
os.makedirs(DRIVE_OUT, exist_ok=True)
print("📁 Outputs will be mirrored to:", DRIVE_OUT)

Mounted at /content/drive
📥 Copying dataset from Drive to local disk...
✅ Copied.
📁 Outputs will be mirrored to: /content/drive/MyDrive/litnet_ablation


In [3]:
# ============================================================
# 2. PATHS & GLOBAL ABLATION CONFIG
# ============================================================
IMG_PATH       = f"{LOCAL_DATASET}/train_images"
MASK_PATH      = f"{LOCAL_DATASET}/train_masks"
TEST_IMG_PATH  = f"{LOCAL_DATASET}/test_images"
TEST_MASK_PATH = f"{LOCAL_DATASET}/test_masks"

ABLATION_DIR = "ablation_results"
CKPT_DIR     = "ablation_ckpts"
os.makedirs(ABLATION_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

IMG_SIZE        = 128
BATCH_SIZE      = 16
EPOCHS          = 30
# Repeat every configuration with matched, independent random seeds.
# Increase RUN_SEEDS (e.g., to 10 runs) if compute resources permit.
RUN_SEEDS = [42, 123, 2026, 3407, 5678]
LR              = 1e-4
VALIDATION_FRACTION = 0.20
EXPERIMENT_VERSION = "reviewed_v4_multirun_optimizer_lr"

for p in [IMG_PATH, MASK_PATH, TEST_IMG_PATH, TEST_MASK_PATH]:
    n = len(os.listdir(p)) if os.path.exists(p) else "MISSING"
    print(f"{p:55s} → {n}")

missing = [p for p in [IMG_PATH, MASK_PATH, TEST_IMG_PATH, TEST_MASK_PATH]
           if not os.path.isdir(p)]
if missing:
    raise FileNotFoundError("Dataset folder(s) missing: " + ", ".join(missing))

/content/data_wound_seg/train_images                    → 2208
/content/data_wound_seg/train_masks                     → 2208
/content/data_wound_seg/test_images                     → 559
/content/data_wound_seg/test_masks                      → 552


In [4]:
# ============================================================
# SYNC HELPER — mirror CSV + checkpoints to Drive
# ============================================================
import shutil

def sync_to_drive():
    """Copy all ablation results & checkpoints to Drive."""
    for f in os.listdir(ABLATION_DIR):
        source = os.path.join(ABLATION_DIR, f)
        if os.path.isfile(source):
            shutil.copy2(source, os.path.join(DRIVE_OUT, f))
    ckpt_out = os.path.join(DRIVE_OUT, "ckpts")
    os.makedirs(ckpt_out, exist_ok=True)
    for f in os.listdir(CKPT_DIR):
        source = os.path.join(CKPT_DIR, f)
        if os.path.isfile(source):
            shutil.copy2(source, os.path.join(ckpt_out, f))
    print(f"💾 Synced {len(os.listdir(ABLATION_DIR))} CSVs + "
          f"{len(os.listdir(CKPT_DIR))} ckpts → {DRIVE_OUT}")

In [5]:
# ============================================================
# RESTORE ONLY RESULTS FROM THE CURRENT EXPERIMENT VERSION
# ============================================================
# Previous CSVs are preserved in a legacy folder, not silently mixed
# with results generated using the corrected evaluation protocol.
legacy_dir = os.path.join(ABLATION_DIR, "legacy_results")
os.makedirs(legacy_dir, exist_ok=True)
for f in os.listdir(ABLATION_DIR):
    local_fp = os.path.join(ABLATION_DIR, f)
    if not f.endswith(".csv") or f == "master.csv":
        continue
    try:
        old = pd.read_csv(local_fp)
        compatible = ("experiment_version" in old.columns and
                      old["experiment_version"].astype(str).eq(EXPERIMENT_VERSION).all())
    except Exception:
        compatible = False
    if not compatible:
        target = os.path.join(legacy_dir, f)
        if os.path.exists(target):
            stem, ext = os.path.splitext(f)
            target = os.path.join(legacy_dir, f"{stem}_{int(time.time())}{ext}")
        shutil.move(local_fp, target)

restored = 0
ckpt_src = os.path.join(DRIVE_OUT, "ckpts") if "DRIVE_OUT" in globals() else None
if os.path.isdir(DRIVE_OUT):
    for f in os.listdir(DRIVE_OUT):
        if not f.endswith(".csv") or f == "master.csv":
            continue
        fp = os.path.join(DRIVE_OUT, f)
        try:
            candidate = pd.read_csv(fp)
            compatible = ("experiment_version" in candidate.columns and
                          candidate["experiment_version"].astype(str).eq(EXPERIMENT_VERSION).all())
        except Exception:
            compatible = False
        if compatible:
            shutil.copy2(fp, os.path.join(ABLATION_DIR, f))
            restored += 1
    if ckpt_src and os.path.isdir(ckpt_src):
        for f in os.listdir(ckpt_src):
            shutil.copy2(os.path.join(ckpt_src, f), os.path.join(CKPT_DIR, f))
print(f"Restored {restored} compatible result CSV(s). Legacy CSVs, if any, are preserved separately.")

Restored 2 compatible result CSV(s). Legacy CSVs, if any, are preserved separately.


In [6]:
# ============================================================
# DATASET ALIGNMENT + FILE INTEGRITY CHECK
# ============================================================
from PIL import Image
from collections import Counter

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")

def make_valid_pairs(img_dir, mask_dir, tag=""):
    """Return readable, basename-matched image/mask path pairs; report exclusions."""
    imgs = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir)
            if f.lower().endswith(IMAGE_EXTS)}
    masks = {os.path.splitext(f)[0]: f for f in os.listdir(mask_dir)
             if f.lower().endswith(IMAGE_EXTS)}
    common = sorted(set(imgs) & set(masks))
    img_only = sorted(set(imgs) - set(masks))
    mask_only = sorted(set(masks) - set(imgs))
    valid, unreadable = [], []

    for name in common:
        ip = os.path.join(img_dir, imgs[name])
        mp = os.path.join(mask_dir, masks[name])
        try:
            with Image.open(ip) as im:
                im.verify()
            with Image.open(mp) as im:
                im.verify()
            valid.append((name, ip, mp))
        except Exception as exc:
            unreadable.append((name, str(exc)))

    print(f"[{tag}] images={len(imgs)} masks={len(masks)} basename_matches={len(common)}")
    print(f"[{tag}] valid_pairs={len(valid)} image_without_mask={len(img_only)} mask_without_image={len(mask_only)} unreadable_pairs={len(unreadable)}")
    if img_only:
        print("  Image without mask (first 10):", img_only[:10])
    if mask_only:
        print("  Mask without image (first 10):", mask_only[:10])
    if unreadable:
        print("  Unreadable pairs (first 10):", unreadable[:10])
    if not valid:
        raise ValueError(f"No valid image-mask pairs found for {tag}.")
    return [(ip, mp) for _, ip, mp in valid]

train_pairs_all = make_valid_pairs(IMG_PATH, MASK_PATH, "TRAIN SOURCE")
test_pairs = make_valid_pairs(TEST_IMG_PATH, TEST_MASK_PATH, "TEST")

# Deterministic split: validation is drawn only from the training directory.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(train_pairs_all))
n_val = max(1, int(round(len(train_pairs_all) * VALIDATION_FRACTION)))
if len(train_pairs_all) - n_val < 1:
    raise ValueError("Not enough valid training pairs for a train/validation split.")
val_idx = set(indices[:n_val].tolist())
train_pairs = [pair for i, pair in enumerate(train_pairs_all) if i not in val_idx]
val_pairs = [pair for i, pair in enumerate(train_pairs_all) if i in val_idx]
print(f"Split sizes: train={len(train_pairs)}, validation={len(val_pairs)}, held-out test={len(test_pairs)}")

[TRAIN SOURCE] images=2208 masks=2208 basename_matches=2208
[TRAIN SOURCE] valid_pairs=2206 image_without_mask=0 mask_without_image=0 unreadable_pairs=2
  Unreadable pairs (first 10): [('wsnet_0035', "cannot identify image file '/content/data_wound_seg/train_images/wsnet_0035.png'"), ('wsnet_1079', "broken PNG file (bad header checksum in b'IDAT')")]
[TEST] images=558 masks=552 basename_matches=552
[TEST] valid_pairs=551 image_without_mask=6 mask_without_image=0 unreadable_pairs=1
  Image without mask (first 10): ['fusc_0594 (1)', 'fusc_0600 (1)', 'fusc_0603 (1)', 'fusc_0605 (1)', 'fusc_0607 (1)', 'fusc_0612 (1)']
  Unreadable pairs (first 10): [('fusc_0882', "broken PNG file (bad header checksum in b'IDAT')")]
Split sizes: train=1765, validation=441, held-out test=551


In [7]:
# ============================================================
# PREPROCESSING & AUGMENTATION
# ============================================================
def preprocess_image(image):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    return tf.cast(image, tf.float32) / 255.0

def preprocess_mask(mask):
    # decode_image returns uint8 for common image formats. Normalize before thresholding.
    mask = tf.image.resize(mask, (IMG_SIZE, IMG_SIZE), method="nearest")
    mask = tf.cast(mask, tf.float32) / 255.0
    return tf.cast(mask >= 0.5, tf.float32)

def augment_image_and_mask(image, mask, seed):
    """Apply identical geometric transforms to image/mask; vary seed by sample."""
    s = tf.cast(seed, tf.int32)
    image = tf.image.stateless_random_flip_left_right(image, s)
    mask  = tf.image.stateless_random_flip_left_right(mask, s)
    s2 = s + tf.constant([1, 0], dtype=tf.int32)
    image = tf.image.stateless_random_flip_up_down(image, s2)
    mask  = tf.image.stateless_random_flip_up_down(mask, s2)
    s3 = s + tf.constant([2, 0], dtype=tf.int32)
    image = tf.image.stateless_random_brightness(image, max_delta=0.1, seed=s3)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask

def _decode_any(path, channels):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=channels, expand_animations=False)
    image.set_shape([None, None, channels])
    return image

def load_image_mask(img_path, mask_path, seed):
    image = preprocess_image(_decode_any(img_path, 3))
    mask = preprocess_mask(_decode_any(mask_path, 1))
    return augment_image_and_mask(image, mask, seed)

In [8]:
# ============================================================
# DATASET BUILDER + EXPLICIT TRAIN / VALIDATION / TEST DATASETS
# ============================================================
def create_dataset(pairs, batch_size=BATCH_SIZE, shuffle=False, augment=False, seed=None):
    seed = SEED if seed is None else int(seed)
    if not pairs:
        raise ValueError("Cannot create a dataset from zero image-mask pairs.")
    image_paths = [p[0] for p in pairs]
    mask_paths = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if shuffle:
        ds = ds.shuffle(len(pairs), seed=seed, reshuffle_each_iteration=True)
    if augment:
        ds = ds.enumerate()
        def _load_aug(index, pair):
            # Unique deterministic seed per sample. Augmentation is reproducible across runs.
            aug_seed = tf.stack([tf.cast(seed, tf.int32), tf.cast(index, tf.int32)])
            return load_image_mask(pair[0], pair[1], aug_seed)
        ds = ds.map(_load_aug, num_parallel_calls=tf.data.AUTOTUNE,
                    deterministic=True)
    else:
        def _load_eval(img_path, mask_path):
            image = preprocess_image(_decode_any(img_path, 3))
            mask = preprocess_mask(_decode_any(mask_path, 1))
            return image, mask
        ds = ds.map(_load_eval, num_parallel_calls=tf.data.AUTOTUNE,
                    deterministic=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(train_pairs, shuffle=True, augment=True)
val_ds   = create_dataset(val_pairs, shuffle=False, augment=False)
test_ds  = create_dataset(test_pairs, shuffle=False, augment=False)

print("Dataset batches:", {
    "train": tf.data.experimental.cardinality(train_ds).numpy(),
    "validation": tf.data.experimental.cardinality(val_ds).numpy(),
    "test": tf.data.experimental.cardinality(test_ds).numpy(),
})

Dataset batches: {'train': np.int64(111), 'validation': np.int64(28), 'test': np.int64(35)}


In [9]:
# ============================================================
# OPTIONAL FILE FORMAT AUDIT
# ============================================================
def scan_formats(folder, tag):
    ext_counter, actual_counter, unreadable = Counter(), Counter(), []
    files = sorted(f for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXTS))
    for filename in files:
        ext_counter[os.path.splitext(filename)[1].lower()] += 1
        try:
            with Image.open(os.path.join(folder, filename)) as im:
                actual_counter[(im.format or "unknown").lower()] += 1
                im.verify()
        except Exception as exc:
            unreadable.append((filename, str(exc)))
    print(f"{tag}: extensions={dict(ext_counter)} actual_formats={dict(actual_counter)} unreadable={len(unreadable)}")
    if unreadable:
        print("Unreadable examples:", unreadable[:10])
    return unreadable

# The pair-integrity check above already excludes unreadable matched pairs.
# This audit is informational and does not silently modify the dataset.
scan_formats(IMG_PATH, "TRAIN IMAGES")
scan_formats(MASK_PATH, "TRAIN MASKS")
scan_formats(TEST_IMG_PATH, "TEST IMAGES")
scan_formats(TEST_MASK_PATH, "TEST MASKS")

TRAIN IMAGES: extensions={'.png': 2208} actual_formats={'png': 2207} unreadable=2
Unreadable examples: [('wsnet_0035.png', "cannot identify image file '/content/data_wound_seg/train_images/wsnet_0035.png'"), ('wsnet_1079.png', "broken PNG file (bad header checksum in b'IDAT')")]
TRAIN MASKS: extensions={'.png': 2208} actual_formats={'png': 2208} unreadable=0
TEST IMAGES: extensions={'.png': 558} actual_formats={'png': 558} unreadable=1
Unreadable examples: [('fusc_0882.png', "broken PNG file (bad header checksum in b'IDAT')")]
TEST MASKS: extensions={'.png': 552} actual_formats={'png': 552} unreadable=0


[]

In [10]:
# ============================================================
# CELL 9 — METRICS & LOSSES
# ============================================================
def dice_loss(y_true, y_pred, smooth=1e-6):
    yt = tf.reshape(y_true, [-1]); yp = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(yt * yp)
    return 1 - (2.*inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt = tf.reshape(y_true, [-1]); yp = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(yt * yp)
    return (2.*inter + smooth) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + smooth)

def bce_only(y_true, y_pred, **kw):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))

def dice_only(y_true, y_pred, **kw):
    return dice_loss(y_true, y_pred)

def combined_loss(y_true, y_pred, alpha=0.5, smooth=1e-6):
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return alpha * bce + (1 - alpha) * dice_loss(y_true, y_pred, smooth)

def combined_alpha(alpha):
    def _loss(y_true, y_pred, **kw):
        return combined_loss(y_true, y_pred, alpha=alpha)
    return _loss

In [11]:
# ============================================================
# CELL 10 — DLU ACTIVATION LAYER
# ============================================================
from tensorflow.keras import layers, models

class DLU(layers.Layer):
    def __init__(self, beta_init=0.0, beta_max=0.01, **kwargs):
        super().__init__(**kwargs)
        self.beta_init = beta_init
        self.beta_max  = beta_max

    def build(self, input_shape):
        self.beta = self.add_weight(
            name="beta", shape=(1,),
            initializer=tf.keras.initializers.Constant(self.beta_init),
            trainable=True,
            constraint=tf.keras.constraints.MinMaxNorm(0.0, self.beta_max))
        super().build(input_shape)

    def call(self, inputs):
        return tf.maximum(inputs, self.beta * inputs)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"beta_init": self.beta_init, "beta_max": self.beta_max})
        return cfg

In [12]:
# ============================================================
# CELL 11 — PARAMETERIZED LitUNet FOR ABLATIONS
# ============================================================
def LitUNet_Ablated(input_size=(IMG_SIZE, IMG_SIZE, 3),
                    filters=(32, 64, 128, 256),
                    dropout_rate=0.4,
                    use_batchnorm=True,
                    activation='DLU',
                    beta_max=0.01,
                    name="LitUNet-Ablation"):
    f1, f2, f3, f4 = filters

    def act(x):
        if activation == 'DLU':
            return DLU(beta_max=beta_max)(x)
        if activation == 'ReLU':
            return layers.ReLU()(x)
        if activation == 'LeakyReLU':
            return layers.LeakyReLU(alpha=0.1)(x)
        raise ValueError(f"Unknown activation: {activation}")

    def conv_block(x, f):
        x = layers.Conv2D(f, 3, padding='same')(x)
        if use_batchnorm: x = layers.BatchNormalization()(x)
        x = act(x)
        x = layers.Conv2D(f, 3, padding='same')(x)
        if use_batchnorm: x = layers.BatchNormalization()(x)
        return act(x)

    inputs = layers.Input(input_size)

    # Encoder
    c1 = conv_block(inputs, f1); p1 = layers.MaxPooling2D(2)(c1)
    c2 = conv_block(p1,     f2); p2 = layers.MaxPooling2D(2)(c2)
    c3 = conv_block(p2,     f3); p3 = layers.MaxPooling2D(2)(c3)

    # Bottleneck
    c4 = conv_block(p3, f4)
    c4 = layers.Dropout(dropout_rate)(c4)

    # Decoder
    u5 = layers.Conv2DTranspose(f3, 2, strides=2, padding='same')(c4)
    c5 = conv_block(layers.concatenate([c3, u5], axis=3), f3)

    u6 = layers.Conv2DTranspose(f2, 2, strides=2, padding='same')(c5)
    c6 = conv_block(layers.concatenate([c2, u6], axis=3), f2)

    u7 = layers.Conv2DTranspose(f1, 2, strides=2, padding='same')(c6)
    c7 = conv_block(layers.concatenate([c1, u7], axis=3), f1)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c7)
    return models.Model(inputs, outputs, name=name)

In [13]:
# ============================================================
# SEGMENTATION METRICS (THRESHOLDED, POOLED PIXEL METRICS)
# ============================================================
def segmentation_metrics(model, ds, threshold=0.5):
    """Compute pooled pixel-level metrics; call on validation or test explicitly."""
    y_true, y_prob = [], []
    for xb, yb in ds:
        pred = model(xb, training=False).numpy()
        y_true.append(yb.numpy().reshape(-1))
        y_prob.append(pred.reshape(-1))
    if not y_true:
        raise ValueError("Evaluation dataset produced no batches.")
    y_true = np.concatenate(y_true) >= 0.5
    y_prob = np.concatenate(y_prob)
    y_pred = y_prob >= threshold

    tp = int(np.sum(y_pred & y_true))
    tn = int(np.sum(~y_pred & ~y_true))
    fp = int(np.sum(y_pred & ~y_true))
    fn = int(np.sum(~y_pred & y_true))
    eps = 1e-7
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    return {"dice": float(dice), "iou": float(iou), "accuracy": float(accuracy),
            "precision": float(precision), "recall": float(recall), "f1": float(f1)}

# Backwards-compatible name; the supplied dataset determines the split evaluated.
full_test_metrics = segmentation_metrics

In [14]:
# ============================================================
# RUN EACH ABLATION CONFIGURATION OVER MATCHED RANDOM SEEDS
# ============================================================
def run_config(config_tag, builder_fn, loss_fn=combined_loss,
               csv_path=None, epochs=EPOCHS, lr=LR, run_seeds=None,
               optimizer_name="Adam", optimizer_kwargs=None,
               reduce_lr_on_plateau=True):
    """Train/reuse one row per seed; return the list of per-seed result rows."""
    global SEED, train_ds
    seeds = list(RUN_SEEDS if run_seeds is None else run_seeds)
    if not seeds:
        raise ValueError("run_seeds must contain at least one seed")

    # Keep a fresh result table and reject CSVs from incompatible experiment versions.
    if csv_path and os.path.exists(csv_path):
        current = pd.read_csv(csv_path)
        compatible = ("experiment_version" in current.columns and
                      current["experiment_version"].astype(str).eq(EXPERIMENT_VERSION).all())
        if not compatible:
            legacy = csv_path + ".legacy"
            if os.path.exists(legacy):
                legacy = csv_path + f".{int(time.time())}.legacy"
            shutil.move(csv_path, legacy)

    rows = []
    for run_seed in seeds:
        SEED = int(run_seed)
        tf.keras.utils.set_random_seed(SEED)
        random.seed(SEED); np.random.seed(SEED)

        # Rebuild the training dataset so shuffle and augmentation use this run's seed.
        train_ds = create_dataset(train_pairs, shuffle=True, augment=True, seed=SEED)
        ckpt = os.path.join(CKPT_DIR, f"{config_tag}_seed{SEED}.weights.h5")

        existing = pd.read_csv(csv_path) if csv_path and os.path.exists(csv_path) else pd.DataFrame()
        if len(existing):
            hit = existing[
                (existing["experiment_version"].astype(str) == EXPERIMENT_VERSION) &
                (existing["config"].astype(str) == config_tag) &
                (pd.to_numeric(existing["seed"], errors="coerce") == SEED)
            ]
        else:
            hit = pd.DataFrame()
        if len(hit) and os.path.exists(ckpt):
            row = hit.iloc[-1].to_dict()
            rows.append(row)
            print(f"Cached: {config_tag}, seed={SEED}, validation Dice={float(row['dice']):.4f}")
            continue
        elif len(hit):
            print(f"Result row exists but checkpoint is missing for {config_tag}, seed={SEED}; retraining.")

        print(f"\n>>> RUN {config_tag} | seed={SEED}")
        tf.keras.backend.clear_session()
        gc.collect()
        # Re-apply the seed after clearing the Keras session.
        tf.keras.utils.set_random_seed(SEED)
        model = builder_fn()
        optimizer_kwargs = dict(optimizer_kwargs or {})
        optimizer_classes = {
            "Adam": tf.keras.optimizers.Adam,
            "SGD": tf.keras.optimizers.SGD,
            "RMSprop": tf.keras.optimizers.RMSprop,
        }
        if optimizer_name not in optimizer_classes:
            raise ValueError(f"Unsupported optimizer: {optimizer_name}. Choose from {list(optimizer_classes)}")
        optimizer = optimizer_classes[optimizer_name](learning_rate=lr, **optimizer_kwargs)
        model.compile(
            optimizer=optimizer,
            loss=loss_fn,
            metrics=["accuracy", dice_coefficient],
        )
        callbacks = [
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                             restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss",
                                               save_best_only=True, save_weights_only=True, verbose=0),
        ]
        # For the learning-rate ablation, keep the assigned LR fixed so the
        # comparison measures the selected initial learning rate rather than
        # interactions with an adaptive LR scheduler.
        if reduce_lr_on_plateau:
            callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=0))
        start = time.time()
        history = model.fit(train_ds, validation_data=val_ds,
                            epochs=epochs, callbacks=callbacks, verbose=1)
        minutes = (time.time() - start) / 60.0
        if not os.path.exists(ckpt):
            raise FileNotFoundError(f"Best checkpoint was not created: {ckpt}")
        model.load_weights(ckpt)
        val_loss, val_accuracy, val_dice_soft = model.evaluate(val_ds, verbose=0)
        val_metrics = segmentation_metrics(model, val_ds)
        row = {
            "experiment_version": EXPERIMENT_VERSION,
            "evaluation_split": "validation",
            "config": config_tag,
            "optimizer": optimizer_name,
            "learning_rate_initial": float(lr),
            "optimizer_kwargs": json.dumps(optimizer_kwargs, sort_keys=True),
            "lr_scheduler": bool(reduce_lr_on_plateau),
            "loss": float(val_loss),
            "accuracy": val_metrics["accuracy"],
            "dice_val_soft": float(val_dice_soft),
            "dice": val_metrics["dice"],
            "iou": val_metrics["iou"],
            "precision": val_metrics["precision"],
            "recall": val_metrics["recall"],
            "f1": val_metrics["f1"],
            "epochs_run": len(history.history["loss"]),
            "minutes": round(minutes, 2),
            "params": int(model.count_params()),
            "seed": SEED,
            "checkpoint": os.path.basename(ckpt),
        }
        print(f"  validation Dice={row['dice']:.4f} IoU={row['iou']:.4f}; "
              f"epochs={row['epochs_run']} time={row['minutes']} min")
        if csv_path:
            existing = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame()
            if len(existing) and ("experiment_version" not in existing.columns or
                                  not existing["experiment_version"].astype(str).eq(EXPERIMENT_VERSION).all()):
                raise ValueError(f"Incompatible CSV still exists: {csv_path}. Move it aside and rerun.")
            if len(existing):
                existing = existing[~((existing["config"].astype(str) == config_tag) &
                                      (pd.to_numeric(existing["seed"], errors="coerce") == SEED))]
            updated = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
            updated.to_csv(csv_path, index=False)
        rows.append(row)
        del model
        gc.collect()
    return rows


In [15]:
# ============================================================
# CELL 14 — SYNC RESULTS & CHECKPOINTS TO DRIVE
# ============================================================
def sync_to_drive():
    """Mirror all ablation CSVs + checkpoints to Drive."""
    for f in os.listdir(ABLATION_DIR):
        source = os.path.join(ABLATION_DIR, f)
        if os.path.isfile(source):
            shutil.copy2(source, os.path.join(DRIVE_OUT, f))

    ckpt_out = os.path.join(DRIVE_OUT, "ckpts")
    os.makedirs(ckpt_out, exist_ok=True)
    for f in os.listdir(CKPT_DIR):
        source = os.path.join(CKPT_DIR, f)
        if os.path.isfile(source):
            shutil.copy2(source, os.path.join(ckpt_out, f))

    n_csv  = len(os.listdir(ABLATION_DIR))
    n_ckpt = len(os.listdir(CKPT_DIR))
    print(f"💾 Synced {n_csv} CSV(s) + {n_ckpt} checkpoint(s) → {DRIVE_OUT}")

In [16]:
# ============================================================
# CELL 15 — SANITY CHECK (forward pass through variants)
# ============================================================
tests = [
    ("ReLU",   dict(activation='ReLU')),
    ("DLU",    dict(activation='DLU')),
    ("no-BN",  dict(activation='DLU', use_batchnorm=False)),
    ("small",  dict(activation='DLU', filters=(16,32,64,128))),
    ("large",  dict(activation='DLU', filters=(64,128,256,512))),
]
x_batch, y_batch = next(iter(train_ds))
print(f"Batch shape: {x_batch.shape}")
for tag, kw in tests:
    m = LitUNet_Ablated(**kw)
    yh = m(x_batch, training=False)
    assert yh.shape == (x_batch.shape[0], IMG_SIZE, IMG_SIZE, 1), \
        f"{tag}: bad output {yh.shape}"
    print(f"✅ {tag:8s} out={tuple(yh.shape)}  params={m.count_params():,}")
    del m; gc.collect()

Batch shape: (16, 128, 128, 3)
✅ ReLU     out=(16, 128, 128, 1)  params=1,931,233
✅ DLU      out=(16, 128, 128, 1)  params=1,931,247
✅ no-BN    out=(16, 128, 128, 1)  params=1,925,615
✅ small    out=(16, 128, 128, 1)  params=484,863
✅ large    out=(16, 128, 128, 1)  params=7,708,623


### Running the multi-run experiment

Run all setup and data cells first, then execute each ablation cell. The first run of each configuration trains once for every seed in `RUN_SEEDS`; rerunning a cell reuses a result only when both its version/seed row and seed-specific checkpoint are present. For a clean full experiment, use a fresh runtime and keep the same `RUN_SEEDS` for every ablation axis. Each seed/configuration run can take several minutes, so the full experiment may take substantially longer than the single-run notebook.

In [17]:
# ============================================================
# ABLATION 1 — ACTIVATION FUNCTION
# ============================================================
csv_act = os.path.join(ABLATION_DIR, "01_activation.csv")
configs_act = [
    ("act_ReLU",      dict(activation='ReLU')),
    ("act_LeakyReLU", dict(activation='LeakyReLU')),
    ("act_DLU",       dict(activation='DLU', beta_max=0.01)),   # baseline
]

for tag, kw in configs_act:
    run_config(tag,
               builder_fn=lambda kw=kw, tag=tag: LitUNet_Ablated(name=tag, **kw),
               csv_path=csv_act)

df_act = pd.read_csv(csv_act).sort_values("dice", ascending=False)
print("\n📊 ABLATION 1 — ACTIVATION\n", df_act.to_string(index=False, float_format="%.4f"))
sync_to_drive()

Cached: act_ReLU, seed=42, validation Dice=0.8313
Cached: act_ReLU, seed=123, validation Dice=0.8151
Cached: act_ReLU, seed=2026, validation Dice=0.8135
Cached: act_ReLU, seed=3407, validation Dice=0.8198
Cached: act_ReLU, seed=5678, validation Dice=0.8256
Cached: act_LeakyReLU, seed=42, validation Dice=0.8258
Cached: act_LeakyReLU, seed=123, validation Dice=0.8212
Cached: act_LeakyReLU, seed=2026, validation Dice=0.8162
Cached: act_LeakyReLU, seed=3407, validation Dice=0.8209
Cached: act_LeakyReLU, seed=5678, validation Dice=0.8136
Cached: act_DLU, seed=42, validation Dice=0.8288
Cached: act_DLU, seed=123, validation Dice=0.8199
Cached: act_DLU, seed=2026, validation Dice=0.8134
Cached: act_DLU, seed=3407, validation Dice=0.8275
Cached: act_DLU, seed=5678, validation Dice=0.8241

📊 ABLATION 1 — ACTIVATION
                experiment_version evaluation_split        config optimizer  learning_rate_initial optimizer_kwargs  lr_scheduler   loss  accuracy  dice_val_soft   dice    iou  preci

In [18]:
# ============================================================
# ABLATION 2 — LOSS FUNCTION
# ============================================================
csv_loss = os.path.join(ABLATION_DIR, "02_loss.csv")
configs_loss = [
    ("loss_BCE",       bce_only),
    ("loss_Dice",      dice_only),
    ("loss_Comb_a0.3", combined_alpha(0.3)),
    ("loss_Comb_a0.5", combined_loss),        # baseline
    ("loss_Comb_a0.7", combined_alpha(0.7)),
]

for tag, fn in configs_loss:
    run_config(tag,
               builder_fn=lambda tag=tag: LitUNet_Ablated(name=tag, activation='DLU'),
               loss_fn=fn,
               csv_path=csv_loss)

df_loss = pd.read_csv(csv_loss).sort_values("dice", ascending=False)
print("\n📊 ABLATION 2 — LOSS\n", df_loss.to_string(index=False, float_format="%.4f"))
sync_to_drive()

Cached: loss_BCE, seed=42, validation Dice=0.8150
Cached: loss_BCE, seed=123, validation Dice=0.8112
Cached: loss_BCE, seed=2026, validation Dice=0.7906
Cached: loss_BCE, seed=3407, validation Dice=0.8070
Cached: loss_BCE, seed=5678, validation Dice=0.8019
Cached: loss_Dice, seed=42, validation Dice=0.8207
Cached: loss_Dice, seed=123, validation Dice=0.8084
Cached: loss_Dice, seed=2026, validation Dice=0.8114
Cached: loss_Dice, seed=3407, validation Dice=0.8261
Cached: loss_Dice, seed=5678, validation Dice=0.8153
Cached: loss_Comb_a0.3, seed=42, validation Dice=0.8154
Cached: loss_Comb_a0.3, seed=123, validation Dice=0.8138
Cached: loss_Comb_a0.3, seed=2026, validation Dice=0.8184
Cached: loss_Comb_a0.3, seed=3407, validation Dice=0.8230
Cached: loss_Comb_a0.3, seed=5678, validation Dice=0.8138
Cached: loss_Comb_a0.5, seed=42, validation Dice=0.8287
Cached: loss_Comb_a0.5, seed=123, validation Dice=0.8111
Cached: loss_Comb_a0.5, seed=2026, validation Dice=0.8165
Cached: loss_Comb_a0.5,

In [19]:
# ============================================================
# ABLATION 3 — DROPOUT RATE
# ============================================================
csv_dr = os.path.join(ABLATION_DIR, "03_dropout.csv")
for dr in [0.0, 0.2, 0.4, 0.6]:
    tag = f"drop_{dr}"
    run_config(tag,
               builder_fn=lambda dr=dr, tag=tag: LitUNet_Ablated(
                   name=tag, dropout_rate=dr, activation='DLU'),
               csv_path=csv_dr)

df_dr = pd.read_csv(csv_dr).sort_values("dice", ascending=False)
print("\n📊 ABLATION 3 — DROPOUT\n", df_dr.to_string(index=False, float_format="%.4f"))
sync_to_drive()


>>> RUN drop_0.0 | seed=42
Epoch 1/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 64s 270ms/step - accuracy: 0.8765 - dice_coefficient: 0.1662 - loss: 0.6238 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0733 - val_loss: 0.6760 - learning_rate: 1.0000e-04
Epoch 2/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9577 - dice_coefficient: 0.2459 - loss: 0.4963 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0665 - val_loss: 0.5928 - learning_rate: 1.0000e-04
Epoch 3/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9595 - dice_coefficient: 0.2944 - loss: 0.4543 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0587 - val_loss: 0.5722 - learning_rate: 1.0000e-04
Epoch 4/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9619 - dice_coefficient: 0.3413 - loss: 0.4168 - val_accuracy: 0.9620 - val_dice_coefficient: 0.1379 - val_loss: 0.5163 - learning_rate: 1.0000e-04
Epoch 5/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9661 - dice_coefficient: 0.3776 - loss: 0.3

In [20]:
# ============================================================
# ABLATION 4 — BATCH NORMALIZATION
# ============================================================
csv_bn = os.path.join(ABLATION_DIR, "04_batchnorm.csv")
for use_bn in [True, False]:
    tag = f"bn_{'on' if use_bn else 'off'}"
    run_config(tag,
               builder_fn=lambda use_bn=use_bn, tag=tag: LitUNet_Ablated(
                   name=tag, use_batchnorm=use_bn, activation='DLU'),
               csv_path=csv_bn)

df_bn = pd.read_csv(csv_bn).sort_values("dice", ascending=False)
print("\n📊 ABLATION 4 — BATCHNORM\n", df_bn.to_string(index=False, float_format="%.4f"))
sync_to_drive()


>>> RUN bn_on | seed=42
Epoch 1/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 43s 187ms/step - accuracy: 0.9047 - dice_coefficient: 0.1726 - loss: 0.6065 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0734 - val_loss: 0.6788 - learning_rate: 1.0000e-04
Epoch 2/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9567 - dice_coefficient: 0.2501 - loss: 0.4951 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0680 - val_loss: 0.6011 - learning_rate: 1.0000e-04
Epoch 3/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9601 - dice_coefficient: 0.3000 - loss: 0.4504 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0607 - val_loss: 0.5687 - learning_rate: 1.0000e-04
Epoch 4/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 44ms/step - accuracy: 0.9625 - dice_coefficient: 0.3503 - loss: 0.4103 - val_accuracy: 0.9620 - val_dice_coefficient: 0.1289 - val_loss: 0.5199 - learning_rate: 1.0000e-04
Epoch 5/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9664 - dice_coefficient: 0.3910 - loss: 0.3785

In [21]:
# ============================================================
# ABLATION 5 — NETWORK WIDTH
# ============================================================
csv_w = os.path.join(ABLATION_DIR, "05_width.csv")
widths = {
    "small": (16, 32, 64, 128),
    "base":  (32, 64, 128, 256),     # baseline
    "large": (64, 128, 256, 512),
}
for key, f in widths.items():
    tag = f"width_{key}"
    run_config(tag,
               builder_fn=lambda f=f, tag=tag: LitUNet_Ablated(
                   name=tag, filters=f, activation='DLU'),
               csv_path=csv_w)

df_w = pd.read_csv(csv_w).sort_values("dice", ascending=False)
print("\n📊 ABLATION 5 — WIDTH\n", df_w.to_string(index=False, float_format="%.4f"))
sync_to_drive()


>>> RUN width_small | seed=42
Epoch 1/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 55s 234ms/step - accuracy: 0.8741 - dice_coefficient: 0.1252 - loss: 0.6562 - val_accuracy: 0.9570 - val_dice_coefficient: 0.0762 - val_loss: 0.6983 - learning_rate: 1.0000e-04
Epoch 2/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9421 - dice_coefficient: 0.1980 - loss: 0.5628 - val_accuracy: 0.9569 - val_dice_coefficient: 0.0835 - val_loss: 0.6219 - learning_rate: 1.0000e-04
Epoch 3/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.9529 - dice_coefficient: 0.2301 - loss: 0.5214 - val_accuracy: 0.9562 - val_dice_coefficient: 0.0877 - val_loss: 0.5807 - learning_rate: 1.0000e-04
Epoch 4/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9557 - dice_coefficient: 0.2651 - loss: 0.4867 - val_accuracy: 0.9627 - val_dice_coefficient: 0.1347 - val_loss: 0.5371 - learning_rate: 1.0000e-04
Epoch 5/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9591 - dice_coefficient: 0.2894 - loss: 

In [22]:
# ============================================================
# ABLATION 6 — DLU BETA UPPER BOUND
# ============================================================
csv_b = os.path.join(ABLATION_DIR, "06_beta.csv")
for bmax in [0.01, 0.1, 0.3]:
    tag = f"beta_{bmax}"
    run_config(tag,
               builder_fn=lambda bmax=bmax, tag=tag: LitUNet_Ablated(
                   name=tag, activation='DLU', beta_max=bmax),
               csv_path=csv_b)

df_b = pd.read_csv(csv_b).sort_values("dice", ascending=False)
print("\n📊 ABLATION 6 — DLU BETA\n", df_b.to_string(index=False, float_format="%.4f"))
sync_to_drive()


>>> RUN beta_0.01 | seed=42
Epoch 1/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 43s 189ms/step - accuracy: 0.9047 - dice_coefficient: 0.1726 - loss: 0.6066 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0734 - val_loss: 0.6790 - learning_rate: 1.0000e-04
Epoch 2/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9567 - dice_coefficient: 0.2499 - loss: 0.4953 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0680 - val_loss: 0.6010 - learning_rate: 1.0000e-04
Epoch 3/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9601 - dice_coefficient: 0.3000 - loss: 0.4504 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0602 - val_loss: 0.5688 - learning_rate: 1.0000e-04
Epoch 4/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9627 - dice_coefficient: 0.3508 - loss: 0.4099 - val_accuracy: 0.9621 - val_dice_coefficient: 0.1321 - val_loss: 0.5177 - learning_rate: 1.0000e-04
Epoch 5/30
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9662 - dice_coefficient: 0.3906 - loss: 0.

## Additional ablations: optimizer and learning rate

These experiments use the same LitUNet-DLU architecture, matched seeds, training/validation split, loss, and early-stopping/checkpoint rules. Optimizer comparisons use the same nominal learning rate (`1e-4`) and the existing `ReduceLROnPlateau` policy. Learning-rate comparisons use Adam at `1e-3`, `1e-4`, and `1e-5`; the scheduler is disabled for this axis so the selected learning rate remains fixed during training. The results are validation-only. Do not inspect the held-out test set to choose these settings.

In [23]:
# ============================================================
# ABLATION 7 — OPTIMIZER
# ============================================================
# Three optimizers × the configured RUN_SEEDS. This may take several hours.
OPT_LR_ABLATION_EPOCHS = 15
csv_opt = os.path.join(ABLATION_DIR, "07_optimizer.csv")
optimizer_configs = [
    ("opt_Adam", "Adam", {}),
    ("opt_SGD", "SGD", {"momentum": 0.9}),
    ("opt_RMSprop", "RMSprop", {}),
]

for tag, opt_name, opt_kwargs in optimizer_configs:
    run_config(
        config_tag=tag,
        builder_fn=lambda tag=tag: LitUNet_Ablated(name=tag, activation="DLU"),
        loss_fn=combined_loss,
        csv_path=csv_opt,
        epochs=OPT_LR_ABLATION_EPOCHS,
        lr=1e-4,
        optimizer_name=opt_name,
        optimizer_kwargs=opt_kwargs,
        reduce_lr_on_plateau=True,
    )

df_opt = pd.read_csv(csv_opt)
df_opt = df_opt[df_opt["experiment_version"].astype(str) == EXPERIMENT_VERSION]
print("\nOPTIMIZER ABLATION — per-seed validation results")
print(df_opt.sort_values(["config", "seed"])[["config", "seed", "dice", "iou", "loss", "minutes"]].to_string(index=False, float_format="%.4f"))
sync_to_drive()


>>> RUN opt_Adam | seed=42
Epoch 1/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 42s 182ms/step - accuracy: 0.9047 - dice_coefficient: 0.1726 - loss: 0.6064 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0735 - val_loss: 0.6791 - learning_rate: 1.0000e-04
Epoch 2/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9567 - dice_coefficient: 0.2501 - loss: 0.4951 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0680 - val_loss: 0.6009 - learning_rate: 1.0000e-04
Epoch 3/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9601 - dice_coefficient: 0.3002 - loss: 0.4503 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0612 - val_loss: 0.5682 - learning_rate: 1.0000e-04
Epoch 4/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9626 - dice_coefficient: 0.3505 - loss: 0.4101 - val_accuracy: 0.9621 - val_dice_coefficient: 0.1327 - val_loss: 0.5176 - learning_rate: 1.0000e-04
Epoch 5/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9664 - dice_coefficient: 0.3911 - loss: 0.3

In [24]:
# ============================================================
# ABLATION 8 — LEARNING RATE (Adam; fixed LR during each run)
# ============================================================
csv_lr = os.path.join(ABLATION_DIR, "08_learning_rate.csv")
lr_configs = [
    ("lr_1e-3", 1e-3),
    ("lr_1e-4", 1e-4),
    ("lr_1e-5", 1e-5),
]

for tag, lr_value in lr_configs:
    run_config(
        config_tag=tag,
        builder_fn=lambda tag=tag: LitUNet_Ablated(name=tag, activation="DLU"),
        loss_fn=combined_loss,
        csv_path=csv_lr,
        epochs=OPT_LR_ABLATION_EPOCHS,
        lr=lr_value,
        optimizer_name="Adam",
        optimizer_kwargs={},
        reduce_lr_on_plateau=False,
    )

df_lr = pd.read_csv(csv_lr)
df_lr = df_lr[df_lr["experiment_version"].astype(str) == EXPERIMENT_VERSION]
print("\nLEARNING-RATE ABLATION — per-seed validation results")
print(df_lr.sort_values(["config", "seed"])[["config", "seed", "learning_rate_initial", "dice", "iou", "loss", "minutes"]].to_string(index=False, float_format="%.4f"))
sync_to_drive()


>>> RUN lr_1e-3 | seed=42
Epoch 1/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 42s 184ms/step - accuracy: 0.9380 - dice_coefficient: 0.2672 - loss: 0.4818 - val_accuracy: 0.1432 - val_dice_coefficient: 0.0881 - val_loss: 6.0083
Epoch 2/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - accuracy: 0.9583 - dice_coefficient: 0.4483 - loss: 0.3424 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0170 - val_loss: 0.5906
Epoch 3/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.9636 - dice_coefficient: 0.5352 - loss: 0.2934 - val_accuracy: 0.9579 - val_dice_coefficient: 0.0083 - val_loss: 0.6227
Epoch 4/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9678 - dice_coefficient: 0.6039 - loss: 0.2514 - val_accuracy: 0.9629 - val_dice_coefficient: 0.2985 - val_loss: 0.4457
Epoch 5/15
111/111 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9722 - dice_coefficient: 0.6542 - loss: 0.2211 - val_accuracy: 0.9755 - val_dice_coefficient: 0.5948 - val_loss: 0.2531
Epoch 6/15
111/111 ━━━━━━━━━━━━━━━━━

In [25]:
# ============================================================
# MASTER ABLATION RESULTS + MEAN/SD SUMMARY
# ============================================================
axis_files = {
    "Activation": csv_act,
    "Loss":       csv_loss,
    "Dropout":    csv_dr,
    "BatchNorm":  csv_bn,
    "Width":      csv_w,
    "DLU β":      csv_b,
    "Optimizer":  csv_opt,
    "Learning rate": csv_lr,
}

master_rows = []
for axis, fp in axis_files.items():
    if os.path.exists(fp):
        d = pd.read_csv(fp)
        d = d[d.get("experiment_version", pd.Series(index=d.index, dtype=str)).astype(str) == EXPERIMENT_VERSION].copy()
        if len(d):
            d["axis"] = axis
            master_rows.append(d)
if not master_rows:
    raise RuntimeError("No results for the current multi-run experiment were found. Run the ablation cells first.")
master = pd.concat(master_rows, ignore_index=True)
master_path = os.path.join(ABLATION_DIR, "master_multirun.csv")
master.to_csv(master_path, index=False)

metric_cols = ["dice", "iou", "precision", "recall", "f1", "loss", "accuracy", "minutes"]
master_summary = master.groupby(["axis", "config"], as_index=False).agg(
    **{f"{m}_mean": (m, "mean") for m in metric_cols},
    **{f"{m}_std": (m, "std") for m in metric_cols},
    n_runs=("seed", "nunique"),
)
summary_path = os.path.join(ABLATION_DIR, "master_multirun_summary.csv")
master_summary.to_csv(summary_path, index=False)
sync_to_drive()

print("\n" + "="*90)
print("MULTI-RUN SUMMARY (mean ± standard deviation across seeds)")
print("="*90)
view = master_summary.copy()
view["Dice mean±SD"] = view.apply(lambda r: f"{r['dice_mean']:.4f} ± {r['dice_std']:.4f}" if pd.notna(r['dice_std']) else f"{r['dice_mean']:.4f} ± NA", axis=1)
view["IoU mean±SD"] = view.apply(lambda r: f"{r['iou_mean']:.4f} ± {r['iou_std']:.4f}" if pd.notna(r['iou_std']) else f"{r['iou_mean']:.4f} ± NA", axis=1)
print(view[["axis", "config", "n_runs", "Dice mean±SD", "IoU mean±SD"]].sort_values(["axis", "dice_mean"], ascending=[True, False]).to_string(index=False))

plt.figure(figsize=(14, 8))
plot_data = master_summary.sort_values("dice_mean", ascending=True)
plt.barh(plot_data["config"], plot_data["dice_mean"], xerr=plot_data["dice_std"].fillna(0), capsize=3)
plt.axvline(x=master_summary.loc[master_summary["config"] == "act_DLU", "dice_mean"].iloc[0],
            color="k", ls="--", lw=1.5, label="Baseline mean (act_DLU)")
plt.title("Ablation Configurations — Mean Validation Dice ± SD")
plt.xlabel("Mean Validation Dice")
plt.xlim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

💾 Synced 11 CSV(s) + 150 checkpoint(s) → /content/drive/MyDrive/litnet_ablation

MULTI-RUN SUMMARY (mean ± standard deviation across seeds)


KeyError: 'dice_mean'

In [ ]:
# ============================================================
# PER-AXIS BAR CHARTS (MEAN VALIDATION DICE + LOSS)
# ============================================================
def plot_axis(summary_df, axis_name):
    d = summary_df[summary_df["axis"] == axis_name].sort_values("dice_mean", ascending=True)
    if d.empty:
        return
    fig, ax = plt.subplots(1, 2, figsize=(14, max(3, 0.6*len(d)+2)))
    bars = ax[0].barh(d["config"], d["dice_mean"], xerr=d["dice_std"].fillna(0), capsize=3)
    ax[0].set_xlim(0, 1); ax[0].set_xlabel("Mean Validation Dice ± SD")
    ax[0].set_title(f"{axis_name} — Dice", fontweight="bold"); ax[0].grid(alpha=0.3, axis="x")
    bars = ax[1].barh(d["config"], d["loss_mean"], xerr=d["loss_std"].fillna(0), capsize=3)
    ax[1].set_xlabel("Mean Validation Loss ± SD")
    ax[1].set_title(f"{axis_name} — Loss", fontweight="bold"); ax[1].grid(alpha=0.3, axis="x")
    plt.tight_layout(); plt.show()

for axis_name in master_summary["axis"].drop_duplicates():
    plot_axis(master_summary, axis_name)

In [ ]:
# ============================================================
# CROSS-AXIS COMPARISON (MEAN VALIDATION METRICS)
# ============================================================
if len(master_summary) > 0:
    metrics = ["dice", "iou", "f1", "precision", "recall"]
    labels  = ["Dice", "IoU", "F1", "Precision", "Recall"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(20, 6), sharey=False)
    for ax, met, lab in zip(axes, metrics, labels):
        d = master_summary.sort_values(f"{met}_mean", ascending=True)
        ax.barh(d["config"], d[f"{met}_mean"], xerr=d[f"{met}_std"].fillna(0), capsize=2)
        ax.set_xlabel(f"Mean {lab} ± SD"); ax.set_title(f"{lab} across runs", fontweight="bold")
        ax.set_xlim(0, 1); ax.grid(alpha=0.3, axis="x")
    plt.tight_layout(); plt.show()

In [26]:
# ============================================================
# SEED-MATCHED PAIRED T-TESTS VS BASELINE (act_DLU)
# ============================================================
def _holm_adjust(p_values):
    """Holm-Bonferroni adjusted p-values, returned in original order."""
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    order = np.argsort(p)
    adjusted = np.empty(n, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (n - rank) * p[idx])
        running_max = max(running_max, value)
        adjusted[idx] = running_max
    return adjusted

def paired_test_vs_baseline(master_df, baseline="act_DLU", metric="dice"):
    if metric not in master_df.columns:
        raise ValueError(f"Metric '{metric}' is not in the result table")
    base = master_df[master_df["config"] == baseline][["seed", metric]].dropna()
    if base["seed"].nunique() < 2:
        print(f"Need at least 2 matched runs for '{baseline}' to run paired tests; found {base['seed'].nunique()}. Skipping.")
        return
    results = []
    for cfg in sorted(master_df["config"].dropna().unique()):
        if cfg == baseline:
            continue
        other = master_df[master_df["config"] == cfg][["seed", metric]].dropna()
        paired = base.merge(other, on="seed", how="inner", suffixes=("_base", "_other"))
        if len(paired) < 2:
            print(f"Skipping {cfg}: only {len(paired)} seed-matched pair(s).")
            continue
        differences = paired[f"{metric}_base"].to_numpy() - paired[f"{metric}_other"].to_numpy()
        if np.allclose(differences, 0.0, atol=1e-12, rtol=0.0):
            print(f"Skipping {cfg}: its paired {metric} values are identical to the baseline (no testable difference).")
            continue
        t_stat, p_val = ttest_rel(paired[f"{metric}_base"], paired[f"{metric}_other"])
        if not np.isfinite(p_val):
            print(f"Skipping {cfg}: paired t-test returned a non-finite p-value.")
            continue
        results.append({"baseline": baseline, "config": cfg, "metric": metric,
                        "n_pairs": len(paired), "t": float(t_stat), "p_raw": float(p_val)})
    if not results:
        print("No configurations had at least two matched runs with the baseline.")
        return
    p_adj = _holm_adjust([r["p_raw"] for r in results])
    for r, adj in zip(results, p_adj):
        r["p_holm"] = float(adj)
        r["significant_holm_0.05"] = bool(adj < 0.05)
    out = pd.DataFrame(results).sort_values("p_holm")
    print(f"\nSeed-matched paired t-tests vs '{baseline}' on {metric}; Holm-adjusted p-values")
    print(out.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
    safe_baseline = re.sub(r"[^A-Za-z0-9_-]+", "_", str(baseline))
    out.to_csv(os.path.join(ABLATION_DIR, f"paired_tests_{metric}_vs_{safe_baseline}.csv"), index=False)
    sync_to_drive()

# Original activation-axis comparison against the DLU baseline.
paired_test_vs_baseline(master, baseline="act_DLU", metric="dice")

# Within-axis comparisons are fairer for optimizer/LR settings because each
# axis has a matched experimental protocol. Each comparison is seed-paired.
optimizer_master = master[master["axis"] == "Optimizer"].copy()
if len(optimizer_master):
    paired_test_vs_baseline(optimizer_master, baseline="opt_Adam", metric="dice")
lr_master = master[master["axis"] == "Learning rate"].copy()
if len(lr_master):
    paired_test_vs_baseline(lr_master, baseline="lr_1e-4", metric="dice")


Seed-matched paired t-tests vs 'act_DLU' on dice; Holm-adjusted p-values
baseline         config metric  n_pairs        t   p_raw  p_holm  significant_holm_0.05
 act_DLU        opt_SGD   dice        5 99.86359 0.00000 0.00000                   True
 act_DLU        lr_1e-5   dice        5 23.21613 0.00002 0.00049                   True
 act_DLU        lr_1e-3   dice        5 11.82497 0.00029 0.00673                   True
 act_DLU       opt_Adam   dice        5  9.44777 0.00070 0.01540                   True
 act_DLU         bn_off   dice        5  9.07673 0.00082 0.01715                   True
 act_DLU        lr_1e-4   dice        5  8.00102 0.00132 0.02647                   True
 act_DLU       loss_BCE   dice        5  6.43148 0.00301 0.05711                  False
 act_DLU    opt_RMSprop   dice        5  5.79805 0.00440 0.07920                  False
 act_DLU    width_small   dice        5  5.73834 0.00457 0.07920                  False
 act_DLU    width_large   dice        5 -4.771

In [27]:
# ============================================================
# FINAL — MULTI-RUN ABLATION SUMMARY
# ============================================================
print("\n" + "="*80)
print("MULTI-RUN ABLATION STUDY COMPLETE")
print("="*80)
print(f"\nResults CSVs : {ABLATION_DIR}/")
print(f"Checkpoints : {CKPT_DIR}/")
print(f"Drive backup: {DRIVE_OUT}/")
print(f"\nConfiguration-seed runs recorded: {len(master)}")
print(f"Unique configurations (all axes): {master['config'].nunique()}")
print(f"Total training time: {master['minutes'].sum():.1f} minutes ({master['minutes'].sum()/60:.2f} hours)")
print("\nHighest mean validation Dice per axis (mean ± SD; validation selection only):")
for axis in master_summary["axis"].unique():
    sub = master_summary[master_summary["axis"] == axis].sort_values("dice_mean", ascending=False)
    if len(sub):
        top = sub.iloc[0]
        sd = f"{top['dice_std']:.4f}" if pd.notna(top['dice_std']) else "NA (one run)"
        print(f"  {axis:12s} → {top['config']:22s} Dice={top['dice_mean']:.4f} ± {sd}; "
              f"IoU={top['iou_mean']:.4f}")
best = master_summary.sort_values("dice_mean", ascending=False).iloc[0]
best_sd = f"{best['dice_std']:.4f}" if pd.notna(best['dice_std']) else "NA (one run)"
print(f"\nHighest mean validation Dice: {best['config']} — {best['dice_mean']:.4f} ± {best_sd}")
print("Use the seed-matched paired-test table and Holm-adjusted p-values; do not infer significance from the means alone.")
print("Evaluate the held-out test set only after final configuration selection.")


MULTI-RUN ABLATION STUDY COMPLETE

Results CSVs : ablation_results/
Checkpoints : ablation_ckpts/
Drive backup: /content/drive/MyDrive/litnet_ablation/

Configuration-seed runs recorded: 130
Unique configurations (all axes): 26
Total training time: 365.7 minutes (6.09 hours)

Highest mean validation Dice per axis (mean ± SD; validation selection only):
  Activation   → act_DLU                Dice=0.8227 ± 0.0063; IoU=0.6989
  BatchNorm    → bn_on                  Dice=0.8192 ± 0.0140; IoU=0.6940
  DLU β        → beta_0.3               Dice=0.8235 ± 0.0085; IoU=0.7000
  Dropout      → drop_0.2               Dice=0.8214 ± 0.0067; IoU=0.6970
  Learning rate → lr_1e-3                Dice=0.7853 ± 0.0089; IoU=0.6466
  Loss         → loss_Comb_a0.5         Dice=0.8198 ± 0.0066; IoU=0.6947
  Optimizer    → opt_Adam               Dice=0.7670 ± 0.0168; IoU=0.6222
  Width        → width_large            Dice=0.8521 ± 0.0109; IoU=0.7424

Highest mean validation Dice: width_large — 0.8521 ± 0.010

## Final held-out test evaluation

After the multi-run experiments finish, choose a configuration using its **mean validation Dice** (not test results). Then evaluate one predeclared checkpoint on the held-out test set. The default below reflects the strongest configuration in the uploaded first-run results (`width_large`), but re-check the new multi-run summary before using it. Set `SELECTED_TEST_SEED` to a seed included in `RUN_SEEDS` and make sure that checkpoint exists.

In [28]:
# OPTIONAL FINAL TEST: run only after selecting a configuration from multi-run validation results.
# Default is width_large based on the first-run results; verify the new mean validation summary first.
SELECTED_CONFIG_TAG = "width_large"
SELECTED_MODEL_KWARGS = {"filters": (64, 128, 256, 512), "activation": "DLU"}
SELECTED_TEST_SEED = 42  # predeclare one run; do not select a seed by looking at test metrics

selected_model = LitUNet_Ablated(name=SELECTED_CONFIG_TAG, **SELECTED_MODEL_KWARGS)
selected_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss=combined_loss,
    metrics=["accuracy", dice_coefficient],
)
selected_ckpt = os.path.join(CKPT_DIR, f"{SELECTED_CONFIG_TAG}_seed{SELECTED_TEST_SEED}.weights.h5")
if not os.path.exists(selected_ckpt):
    raise FileNotFoundError(
        f"Checkpoint not found: {selected_ckpt}. Run this configuration for seed {SELECTED_TEST_SEED} first."
    )
selected_model.load_weights(selected_ckpt)
final_test_metrics = segmentation_metrics(selected_model, test_ds)
print(f"Held-out test metrics for pre-selected configuration {SELECTED_CONFIG_TAG}, seed={SELECTED_TEST_SEED}:")
print(pd.Series(final_test_metrics).to_string())

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 158 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Held-out test metrics for pre-selected configuration width_large, seed=42:
dice         0.804862
iou          0.673446
accuracy     0.995125
precision    0.869422
recall       0.749226
f1           0.804861
